In [1]:
import glob
import os
import shutil
import socket
import sys
from pathlib import Path

from pyspark import SparkContext
from pyspark.sql import SparkSession

# Ajuste do sys.path para execucao fora do kernel Jupyter padrao
_spark_home = os.environ.get("SPARK_HOME") or "/usr/local/spark"
_spark_py = os.path.join(_spark_home, "python")
if os.path.isdir(_spark_py) and _spark_py not in sys.path:
    sys.path.insert(0, _spark_py)
_py4j_candidates = sorted(glob.glob(os.path.join(_spark_home, "python", "lib", "py4j-*-src.zip")))
if _py4j_candidates:
    _py4j_zip = _py4j_candidates[-1]
    if _py4j_zip not in sys.path:
        sys.path.insert(0, _py4j_zip)

# JAVA_HOME portavel (evita hardcode de Mac)
if not os.environ.get("JAVA_HOME"):
    ms = Path(r"C:\\Program Files\\Microsoft")
    if ms.is_dir():
        for jdk in sorted(ms.glob("jdk-*-hotspot"), reverse=True):
            if (jdk / "bin" / "java.exe").is_file():
                os.environ["JAVA_HOME"] = str(jdk)
                break
    if not os.environ.get("JAVA_HOME"):
        java_exe = shutil.which("java")
        if java_exe:
            jp = Path(java_exe).resolve()
            if jp.parent.name.lower() == "bin":
                os.environ["JAVA_HOME"] = str(jp.parent.parent)
if not os.environ.get("JAVA_HOME"):
    raise RuntimeError(
        "Defina JAVA_HOME para um JDK 17+ (ex.: Microsoft OpenJDK)."
    )

# Aceita SPARK_MASTER ou JUPYTER_SPARK_MASTER; default local[4]
_spark_master_raw = (
    os.environ.get("SPARK_MASTER") or os.environ.get("JUPYTER_SPARK_MASTER") or ""
)
spark_master = (_spark_master_raw or "").strip() or "local[4]"
if not spark_master.lower().startswith("local") and spark_master.startswith("spark://"):
    try:
        hostport = spark_master[len("spark://") :].split("/", 1)[0]
        host = hostport.rsplit(":", 1)[0]
        socket.gethostbyname(host)
    except OSError:
        print(f"[aviso] SPARK_MASTER={spark_master!r} nao resolve aqui; usando local[4].")
        spark_master = "local[4]"


def _reset_spark_if_stale() -> None:
    try:
        inst = getattr(SparkSession, "_instantiatedSession", None)
        if inst is not None:
            sc = getattr(inst, "_sc", None)
            if sc is not None:
                try:
                    sc.stop()
                except Exception:
                    pass
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
    except Exception:
        pass
    try:
        from pyspark.sql.context import SQLContext

        SQLContext._instantiatedContext = None
    except Exception:
        pass
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkContext._active_spark_context = None


_reset_spark_if_stale()
try:
    spark.stop()
except Exception:
    pass
_reset_spark_if_stale()

_is_local_master = str(spark_master).lower().startswith("local")
_driver_mem = os.environ.get("SPARK_DRIVER_MEMORY") or ("3g" if _is_local_master else "4g")
_shuffle_parts = os.environ.get("SPARK_SQL_SHUFFLE_PARTITIONS", "8")
_default_par = os.environ.get("SPARK_DEFAULT_PARALLELISM", "4")

builder = (
    SparkSession.builder.appName("T032-LinearRegression")
    .master(spark_master)
    .config("spark.driver.memory", _driver_mem)
    .config("spark.sql.shuffle.partitions", _shuffle_parts)
    .config("spark.default.parallelism", _default_par)
)

if _is_local_master:
    _max_result = os.environ.get("SPARK_DRIVER_MAX_RESULT_SIZE", "768m")
    builder = builder.config("spark.driver.maxResultSize", _max_result)
else:
    driver_host = os.environ.get("SPARK_DRIVER_HOST", "spark-notebook")
    _exec_mem = os.environ.get("SPARK_EXECUTOR_MEMORY", "1g")
    _exec_cores = os.environ.get("SPARK_EXECUTOR_CORES", "1")
    _cores_max = os.environ.get("SPARK_CORES_MAX", "4")
    builder = (
        builder.config("spark.driver.host", driver_host)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.executor.memory", _exec_mem)
        .config("spark.executor.cores", _exec_cores)
        .config("spark.cores.max", _cores_max)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.network.timeout", "600s")
        .config("spark.executor.heartbeatInterval", "120s")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SPARK_MASTER:", spark_master)
print("Spark versao:", spark.version)
print("spark.driver.memory (pedido):", _driver_mem)
if _is_local_master:
    print("[info] Modo local: o Spark Master :8080 nao exibira esta app.")
else:
    print("spark.driver.host:", os.environ.get("SPARK_DRIVER_HOST", "spark-notebook"))

SPARK_MASTER: spark://spark-master:7077
Spark versao: 3.2.1
spark.driver.memory (pedido): 3g
spark.driver.host: spark-notebook


In [2]:
# Imports
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# SparkSession ja inicializada na celula anterior
print("Spark rodando! Versao:", spark.version)

Spark rodando! Versao: 3.2.1


In [3]:
# Carregar os Dados
# Em cluster (spark://…), executores NAO veem caminhos relativos ao notebook (`../data`).
# Use o mesmo caminho montado no compose: `/dataset` ou `REPO_ROOT/data`.
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

_candidate_parquets = [
    Path("/dataset/Indian_Weather_Dataset.parquet"),
    REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet",
    Path("../data/Indian_Weather_Dataset.parquet").resolve(),
]
PARQUET_PATH = next((p for p in _candidate_parquets if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError(
        "Parquet nao encontrado. Confirme ./data no repo ou montagem /dataset no Docker."
    )

print("PARQUET_PATH:", PARQUET_PATH)
df = spark.read.parquet(str(PARQUET_PATH))
print(f"Total de registros: {df.count()}")

# Divide os dados em Treino e Teste
df_treino, df_teste = df.randomSplit([0.7, 0.3], seed=42)
print(f"Registros de Treino: {df_treino.count()}")
print(f"Registros de Teste: {df_teste.count()}")

PARQUET_PATH: /dataset/Indian_Weather_Dataset.parquet
Total de registros: 46082160
Registros de Treino: 32259408
Registros de Teste: 13822752


In [4]:
colunas_features = [
    "humidity_pct", "pressure_hPa", "dew_point_C", 
    "solar_radiation_Wm2", "cloud_cover_pct", "wind_speed_ms", 
    "wind_dir_sin", "wind_dir_cos", "cape", "et0_mm", "precip_mm"
]

#Agrupar as features
assembler = VectorAssembler(inputCols=colunas_features, outputCol="raw_features")

#StandardScaller
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                      withStd=True, withMean=True)

#Linear_regression
lr = LinearRegression(featuresCol="scaled_features", labelCol="temperature_C", 
                      regParam=0.1, elasticNetParam=0.0, solver="auto")

#Pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])
print("Pipeline configurada com sucesso")

Pipeline configurada com sucesso


In [5]:
print("Treino")

# Treina o modelo
modelo_treinado = pipeline.fit(df_treino)

# Aplica o modelo nos dados
previsoes = modelo_treinado.transform(df_teste)
evaluator_rmse = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="mae")

rmse = evaluator_rmse.evaluate(previsoes)
mae = evaluator_mae.evaluate(previsoes)

print(f"\n--- RESULTADOS DO MODELO ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

Treino

--- RESULTADOS DO MODELO ---
RMSE: 1.6435
MAE:  1.1371


In [6]:
# Extrai o modelo do Pipeline
lr_model = modelo_treinado.stages[-1]

print("Intercepto (Temperatura Base):", round(lr_model.intercept, 4))
print("\nCoeficientes gerados pela Regularização Ridge:")

for feature, coef in zip(colunas_features, lr_model.coefficients):
    print(f"- {feature}: {round(coef, 4)}")

Intercepto (Temperatura Base): 23.6115

Coeficientes gerados pela Regularização Ridge:
- humidity_pct: -5.6658
- pressure_hPa: 0.3989
- dew_point_C: 6.7973
- solar_radiation_Wm2: -2.798
- cloud_cover_pct: 0.2588
- wind_speed_ms: -0.2243
- wind_dir_sin: -0.1112
- wind_dir_cos: -0.0769
- cape: 0.0
- et0_mm: 3.605
- precip_mm: 0.1435
